# NovaMart Data Preparation

**Name:** Jazeel Ahmed

**Version:** 4

This notebook cleans and prepares the NovaMart dataset for the Streamlit dashboard.


In [19]:
import pandas as pd
import numpy as np
import glob
import os

In [20]:
# Read all monthly CSV files
file_list = glob.glob("../data/monthly_orders/*.csv")

# Read and combine all files
df = pd.concat([pd.read_csv(file) for file in file_list], ignore_index=True)

# Check the data
df.head()

,order_id,order_date,ship_date,customer_id,product_id,region,category,sub_category,quantity,sales,discount,profit
0,NM202401461113,2024-01-27,2024-02-01,CU0049,PR522,central,office supplies,Storage,1,$41.69,0.00,19.67
1,NM202401729507,2024-01-29,2024-01-31,CU0229,PR507,CENTRAL,Technology,Machines,4,$170.76,0.00,91.36
2,NM202401472052,2024-01-08,2024-01-13,CU0108,PR501,east,technology,Phones,1,$728.04,0.30,14.63
3,NM202401384236,2024-01-22,2024-01-27,CU0220,PR503,CENTRAL,Technology,Phones,7,$921.60,0.15,NaN
4,NM202401487969,"Jan 04, 2024","Jan 07, 2024",CU0213,PR501,SOUTH,technology,Phones,2,"$2,080.12",0.00,653.30


In [21]:
# Check dataset shape
print("Rows and Columns:", df.shape)

# Check column names
print(df.columns)

Rows and Columns: (6107, 12)
Index(['order_id', 'order_date', 'ship_date', 'customer_id', 'product_id',
       'region', 'category', 'sub_category', 'quantity', 'sales', 'discount',
       'profit'],
      dtype='object')


In [22]:
# Check number of rows before removing duplicates
print("Before:", df.shape)

# Remove duplicate rows
df = df.drop_duplicates()

# Check number of rows after removing duplicates
print("After:", df.shape)

Before: (6107, 12)
After: (6059, 12)


In [23]:
df["sales"].head()

0       $41.69
1      $170.76
2      $728.04
3      $921.60
4    $2,080.12
Name: sales, dtype: object

In [24]:
# Clean sales column
df["sales"] = (
    df["sales"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Check result
df["sales"].head()

0      41.69
1     170.76
2     728.04
3     921.60
4    2080.12
Name: sales, dtype: float64

In [25]:
df["sales"].dtype

dtype('float64')

In [26]:
df[["order_date", "ship_date"]].head()

,order_date,ship_date
0,2024-01-27,2024-02-01
1,2024-01-29,2024-01-31
2,2024-01-08,2024-01-13
3,2024-01-22,2024-01-27
4,"Jan 04, 2024","Jan 07, 2024"


In [27]:
# Convert mixed date formats
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    format="mixed",
    errors="coerce"
)

In [28]:
print("Order Date Missing:", df["order_date"].isna().sum())
print("Ship Date Missing:", df["ship_date"].isna().sum())

Order Date Missing: 0
Ship Date Missing: 0


In [29]:
df[["order_date", "ship_date"]].dtypes

order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object

In [30]:
print(df["order_date"].isna().sum())
print(df["ship_date"].isna().sum())

0
0


In [31]:
print(df["order_date"].astype(str).head(20))

0     2024-01-27
1     2024-01-29
2     2024-01-08
3     2024-01-22
4     2024-01-04
5     2024-01-02
6     2024-01-20
7     2024-01-30
8     2024-01-09
9     2024-01-16
10    2024-01-11
11    2024-01-18
12    2024-01-27
13    2024-01-12
14    2024-01-17
15    2024-01-25
16    2024-01-09
17    2024-01-04
18    2024-01-18
19    2024-01-20
Name: order_date, dtype: object


In [32]:
df.loc[df["order_date"].isna() == False, "order_date"].head(20)


0    2024-01-27
1    2024-01-29
2    2024-01-08
3    2024-01-22
4    2024-01-04
5    2024-01-02
6    2024-01-20
7    2024-01-30
8    2024-01-09
9    2024-01-16
10   2024-01-11
11   2024-01-18
12   2024-01-27
13   2024-01-12
14   2024-01-17
15   2024-01-25
16   2024-01-09
17   2024-01-04
18   2024-01-18
19   2024-01-20
Name: order_date, dtype: datetime64[ns]

In [33]:
print("Order Date Missing:", df["order_date"].isna().sum())
print("Ship Date Missing:", df["ship_date"].isna().sum())

Order Date Missing: 0
Ship Date Missing: 0


In [34]:
# Original date values that failed to parse
temp = pd.concat([pd.read_csv(f) for f in glob.glob("../data/monthly_orders/*.csv")], ignore_index=True)

# Show some unique order_date values
print(temp["order_date"].dropna().unique()[:30])

['2024-01-27' '2024-01-29' '2024-01-08' '2024-01-22' 'Jan 04, 2024'
 '2024-01-02' '2024-01-20' '30-Jan-2024' '09-Jan-2024' '2024-01-16'
 '2024-01-11' '2024-01-18' '2024-01-12' '2024-01-17' '2024-01-25'
 '2024-01-09' '2024-01-04' '14-Jan-2024' '24-Jan-2024' '2024-01-10'
 '2024-01-03' '2024-01-21' '2024-01-31' '2024-01-13' '22-Jan-2024'
 '2024-01-23' '2024-01-19' '2024-01-30' '2024-01-26' '2024-01-06']


In [35]:
print("Discount Missing:", df["discount"].isna().sum())
print("Profit Missing:", df["profit"].isna().sum())

Discount Missing: 241
Profit Missing: 179


In [36]:
# Fill missing discount with 0
df["discount"] = df["discount"].fillna(0)

In [37]:
df["profit"].head(10)

0     19.67
1     91.36
2     14.63
3       NaN
4    653.30
5     76.79
6    389.52
7    157.75
8     33.08
9     16.28
Name: profit, dtype: float64

In [38]:
# Fill missing profit with 0
df["profit"] = df["profit"].fillna(0)

In [39]:
print("Discount Missing:", df["discount"].isna().sum())
print("Profit Missing:", df["profit"].isna().sum())

Discount Missing: 0
Profit Missing: 0


In [40]:
# Read lookup files
product_df = pd.read_csv("../data/product_lookup.csv")
customer_df = pd.read_csv("../data/customer_lookup.csv")

# Check first 5 rows
product_df.head()

,product_id,product_name,category,sub_category,unit_cost,list_price
0,PR501,Galaxy S,Technology,Phones,713.41,1040.06
1,PR502,Pixel Mini,Technology,Phones,162.44,317.20
2,PR503,iView 11,Technology,Phones,98.09,154.89
3,PR504,USB Hub,Technology,Accessories,426.59,727.92
4,PR505,BT Mouse,Technology,Accessories,112.52,194.24


In [41]:
customer_df.head()

,customer_id,customer_name,segment,region,city
0,CU0001,Taylor Hunt,Consumer,South,Atlanta
1,CU0002,Alex Lee,Consumer,West,Seattle
2,CU0003,Riley Roy,Consumer,East,Boston
3,CU0004,Jamie Roy,Consumer,South,Atlanta
4,CU0005,Jamie Fox,Home Office,East,New York


In [42]:
print(product_df.columns)
print(customer_df.columns)

Index(['product_id', 'product_name', 'category', 'sub_category', 'unit_cost',
       'list_price'],
      dtype='object')
Index(['customer_id', 'customer_name', 'segment', 'region', 'city'], dtype='object')


In [43]:
# Merge product lookup
df = df.merge(product_df, on="product_id", how="left")

In [44]:
# Merge customer lookup
df = df.merge(customer_df, on="customer_id", how="left")

In [45]:
print(df.shape)
df.head()

(6059, 21)


,order_id,order_date,ship_date,customer_id,product_id,region_x,category_x,sub_category_x,quantity,sales,...,profit,product_name,category_y,sub_category_y,unit_cost,list_price,customer_name,segment,region_y,city
0,NM202401461113,2024-01-27,2024-02-01,CU0049,PR522,central,office supplies,Storage,1,41.69,...,19.67,File Cabinet,Office Supplies,Storage,22.02,41.69,Sam Bell,Corporate,Central,Dallas
1,NM202401729507,2024-01-29,2024-01-31,CU0229,PR507,CENTRAL,Technology,Machines,4,170.76,...,91.36,LaserJet 100,Technology,Machines,19.85,42.69,Drew Bell,Corporate,Central,Chicago
2,NM202401472052,2024-01-08,2024-01-13,CU0108,PR501,east,technology,Phones,1,728.04,...,14.63,Galaxy S,Technology,Phones,713.41,1040.06,Sam Dean,Consumer,East,New York
3,NM202401384236,2024-01-22,2024-01-27,CU0220,PR503,CENTRAL,Technology,Phones,7,921.60,...,0.00,iView 11,Technology,Phones,98.09,154.89,Drew Dean,Consumer,Central,Dallas
4,NM202401487969,2024-01-04,2024-01-07,CU0213,PR501,SOUTH,technology,Phones,2,2080.12,...,653.30,Galaxy S,Technology,Phones,713.41,1040.06,Drew Wood,Consumer,South,Atlanta


In [46]:
# Remove old columns
df.drop(
    columns=["region_x", "category_x", "sub_category_x"],
    inplace=True
)

# Rename lookup columns
df.rename(
    columns={
        "region_y": "region",
        "category_y": "category",
        "sub_category_y": "sub_category"
    },
    inplace=True
)

In [47]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'customer_id', 'product_id',
       'quantity', 'sales', 'discount', 'profit', 'product_name', 'category',
       'sub_category', 'unit_cost', 'list_price', 'customer_name', 'segment',
       'region', 'city'],
      dtype='object')


In [48]:
# Calculate shipping days
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

In [49]:
df[["order_date", "ship_date", "shipping_days"]].head()

,order_date,ship_date,shipping_days
0,2024-01-27,2024-02-01,5
1,2024-01-29,2024-01-31,2
2,2024-01-08,2024-01-13,5
3,2024-01-22,2024-01-27,5
4,2024-01-04,2024-01-07,3


In [50]:
# Create month and year columns
df["month"] = df["order_date"].dt.month_name()
df["year"] = df["order_date"].dt.year

In [51]:
df[["order_date", "month", "year"]].head()

,order_date,month,year
0,2024-01-27,January,2024
1,2024-01-29,January,2024
2,2024-01-08,January,2024
3,2024-01-22,January,2024
4,2024-01-04,January,2024


In [52]:
# Create high value order column
df["high_value_order"] = np.where(df["sales"] > 1000, "Yes", "No")

In [53]:
df[["sales", "high_value_order"]].head(10)

,sales,high_value_order
0,41.69,No
1,170.76,No
2,728.04,No
3,921.60,No
4,2080.12,Yes
5,195.03,No
6,1046.08,Yes
7,495.31,No
8,303.63,No
9,45.04,No


In [54]:
# Save cleaned dataset
df.to_csv("../data/cleaned_novamart_data.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [55]:
import os

os.path.exists("../data/cleaned_novamart_data.csv")

True